In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
from mly.datatools import DataPod, DataSet, generator
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [ ]:
fs=1024
duration=1
window_size=16
detectors=['I']
n_samples=4000
seed=19


In [7]:
wnb_injections = DataSet.load('wnb_4000.pkl')
glitch_injections = DataSet.load('gengli_blips_4000.pkl')

print(len(wnb_injections), len(glitch_injections))

4000 4000


In [8]:
np.random.seed(seed)

def to1d(data):
    return np.asarray(data).squeeze().astype(np.float32)


def place_glitch(g, mode="random"):
    g = to1d(g)
    if mode == "random":
        return np.roll(g, np.random.randint(0, len(g)))
    elif mode == "center":
        peak = int(np.argmax(np.abs(g)))
        target = len(g)//2
        return np.roll(g, target - peak)
    else:
        raise ValueError
    

def scale_by_norm(g, target_ratio):
    return g * (target_ratio / (np.linalg.norm(g) + 1e-12))

combined_pods = []

for i in range(n_samples):
    wnb = to1d(wnb_injections[i].strain)
    glitch = place_glitch(to1d(glitch_injections[i].strain), mode="random")
    w_strength = np.random.uniform(1.0, 3.0)  
    g_strength = np.random.uniform(0.3, 1.5)
    
    wnb_scaled = scale_by_norm(wnb, w_strength)
    glitch_scaled = scale_by_norm(glitch, g_strength)

    combined = wnb_scaled + glitch_scaled

    pod = DataPod(
        strain=combined,
        fs=fs,
        detectors=detectors,
        labels = {'type': 'signal', 'wf':'signaland glitch', 'wnb_strength': w_strength, 'glitch_strength': g_strength})
    combined_pods.append(pod)

wnb_glitch = DataSet(combined_pods, name="wnb_glitch")
wnb_glitch.save('wnb_glitch_4000.pkl')


ValueError: operands could not be broadcast together with shapes (164,) (234,) 

In [ ]:
noise_wnb_glitch = []
for i in range(n_samples):
    ds = generator(
        duration=duration,
        fs=fs,
        detectors=detectors,
        size=1,
        labels={'class':1}
        backgroung
    )